# 03 · Phenology metrics per year and topographic zone

Loads the cached Sentinel-2 statistics, filters dates by quality and derives the autumn phenology of every year and zone: the start, peak and end of the yellowing season, estimated with several methods from the literature and compared in section 7.

- **Settings:** `config.py` (years, date filters, smoothing, fit and season-date settings)
- **Reads:**
  - `cache/<AOI>_zones/` (from `01_download_sentinel2`)
  - `cache/validation/<AOI>/` (from `01b_download_validation`, optional)
- **Writes** to `results/<AOI>/`:
  - `season_dates.csv`: start / mid / peak / end of every method, index, zone and year, with quality columns
  - `season_method_comparison.csv`: the comparison of section 7
  - `eos_by_zone.csv` and `logistic_by_zone.csv`: inputs of `04_climate_analysis`
  - `zones.csv`, `topography_effects.csv`, `eos_projection.csv`
  - `ndvi_timeseries_all_dates.csv`, `ndvi_timeseries_clean.csv` and `ndvi_monthly.csv`

Years whose autumn is not fully cached yet (`SEASON_COMPLETE_DOY`, e.g. the current year) are shown as preliminary in the plots but left out of all tables, trends and models.

Runs offline in one to a few minutes (most of it the bootstrap, `N_BOOT`). Sources are listed at the end.

## Yellowing season: start, peak and end

The aim is three dates per year and zone:

| Date | Meaning | Signal | Candidates (section 5) | Checked against (section 7) |
|---|---|---|---|---|
| **Start** | needles begin to lose chlorophyll | greenness (GRVI, GCC, NDVI) leaves its summer level | `start`: 90 % of the autumn amplitude left | order, data support |
| **Peak** | canopy most yellow: chlorophyll gone, needles still on | yellowness (RCC, NDYI, PSRI) at its maximum; greenness half-way down | `peak`: maximum of a yellowness index; `mid`: 50 % of the greenness decline | MeteoSwiss needle colouring 50 % |
| **End** | needles have fallen | NDVI reaches its post-season level | `end`: 15 % of the amplitude left | MeteoSwiss needle drop 50 %, HR-VPP end of season |

- **Why greenness for the start and NDVI for the end:** the change from green to yellow shows mainly in the visible bands (green falls relative to red; Motohka et al. 2010). Yellow needles still reflect NIR strongly, so NDVI drops mostly with needle fall.
- **Thresholds:** 90 / 50 / 15 % of the amplitude follow MODIS MCD12Q2 (senescence / greendown midpoint / dormancy; Gray et al. 2019) and HR-VPP (end of season at 15 %; Tian et al. 2021).
- **Expected:** start < peak < end in every year and zone; earlier at higher elevation.
- **Limits:** after mid-October there are few clear observations (low sun turns north slopes SCL "dark", clouds, early snow). Every date therefore carries quality columns (section 5f), and the panel (section 6) shows the observations behind it.

In [ ]:
from datetime import date, timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings

from scipy import sparse
from scipy.optimize import OptimizeWarning, curve_fit
from scipy.sparse.linalg import spsolve
from scipy.stats import kendalltau, linregress, theilslopes
from statsmodels.tsa.seasonal import STL

from config import *
from phenology_utils import clean_frame, incomplete_years, index_frame, load_stats, load_zone_table

INDEX_COLORS = {"NDVI": "darkgreen", "GRVI": "darkorange", "PSRI": "purple", "GCC": "yellowgreen",
                "RCC": "firebrick", "NDYI": "goldenrod"}
INDEX_MARKERS = {"NDVI": "o", "GRVI": "s", "PSRI": "^", "GCC": "D", "RCC": "v", "NDYI": "P"}
ASPECT_COLORS = {"N": "tab:blue", "S": "tab:orange", "flat": "grey"}
print(f"AOI: {AOI_NAME} | years {FIRST_YEAR}-{LAST_YEAR} | results: {OUT_DIR}")

## 1. Load and filter the cached statistics

A date is kept for a zone if at least `MIN_VALID_FRACTION` of the zone's pixels are clear (and it passes `MAX_SCENE_CLOUD` / `MAX_SCL_FRACTION`). `df`, `df_clean` and so on refer to the whole AOI (zone `"all"`). `clean_by_zone[index][zone]` holds the cleaned median series of every zone. `INCOMPLETE` lists the years whose cache ends before `SEASON_COMPLETE_DOY`. Indices of `INDICES` that the cache lacks (e.g. GCC / RCC / NDYI before `01` was re-run) are skipped.

In [ ]:
stats_all = load_stats()
zone_table = load_zone_table()
ZONES = list(zone_table.index)   # "all" first, then elevation band x aspect
INCOMPLETE = incomplete_years(stats_all)
if INCOMPLETE:
    print(f"Autumn not fully cached (preliminary, left out of tables and models): {INCOMPLETE}")

missing = [n for n in INDICES if f"{n.lower()}_median" not in stats_all or stats_all[f"{n.lower()}_median"].isna().all()]
if missing:
    print(f"Not in the cache, skipped: {missing} (re-run 01_download_sentinel2 to add them)")
INDICES = [n for n in INDICES if n not in missing]

frames = {name: index_frame(stats_all, name.lower()) for name in INDICES}
clean = {name: clean_frame(stats_all, name.lower()) for name in INDICES}
df, df_clean = frames["NDVI"], clean["NDVI"]
for name in INDICES:
    print(f"{name}: {len(clean[name])} of {len(frames[name])} dates kept")

clean_by_zone = {name: {zone: clean_frame(stats_all, name.lower(), zone)[f"{name.lower()}_median"] for zone in ZONES}
                 for name in INDICES}
display(zone_table[["n_pixels", "elev_mean", "slope_mean", "northness_mean"]].round(2).join(
    pd.DataFrame({name: {z: len(s) for z, s in by_zone.items()} for name, by_zone in clean_by_zone.items()})
      .add_prefix("dates_")))

## 2. Composite and smooth (whole AOI)

Shows monthly median composites and a centred moving average (kept for display and as the baseline EOS method of section 3; section 5 uses methods that work on the observations directly) (`SMOOTH_WINDOW` × 5 days) on a regular 5-day grid, with gaps of up to 30 days interpolated.

In [ ]:
def smooth(series):
    regular = series.resample("5D").mean().interpolate(method="time", limit=6)
    rolled = regular.rolling(SMOOTH_WINDOW, center=True, min_periods=3).mean()
    return rolled.where(regular.notna())


medians = {name: clean[name][f"{name.lower()}_median"] for name in INDICES}
monthly = {name: s.resample("MS").median() for name, s in medians.items()}
smoothed = {name: smooth(s) for name, s in medians.items()}

fig, ax = plt.subplots(figsize=(12, 4))
for name, s in smoothed.items():
    ax.plot(s.index, s, lw=2, color=INDEX_COLORS[name], label=name)
ax.axhline(0, color="black", lw=0.5)
ax.set_ylabel("index value")
ax.grid(alpha=0.3)
ax.legend(loc="upper left")
ax.set_title(f"{', '.join(INDICES)} ({SMOOTH_WINDOW * 5}-day moving average)")
plt.show()

ylims = {"NDVI": (-0.1, 1), "GRVI": (-1, 1)}
for name in INDICES:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.scatter(frames[name].index, frames[name][f"{name.lower()}_median"], s=10, c="lightgrey", label="all dates")
    ax.scatter(medians[name].index, medians[name], s=14, label="clear dates")
    ax.plot(monthly[name].index, monthly[name], "o--", label="monthly median")
    ax.plot(smoothed[name].index, smoothed[name], lw=2, label=f"{SMOOTH_WINDOW * 5}-day moving average")
    ax.set_ylabel(name)
    if name in ylims:
        ax.set_ylim(*ylims[name])
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_title(name)
    plt.show()

## 3. End of season (EOS) from smoothed NDVI

For every year, EOS is the first date after the year's peak at which the smoothed NDVI drops to
- **fixed:** `EOS_FIXED_THRESHOLD`, or
- **midpoint:** halfway between the peak and the minimum after the peak.

Years with fewer than `MIN_POINTS_PER_YEAR` values, or where no crossing occurs (e.g. snow hides the decline), get no EOS. `eos_table` covers all zones and complete years; `eos_table_all` also has the preliminary years. The plot shows the whole AOI, with a linear trend in days per year.

In [ ]:
def first_crossing(post_peak, threshold):
    below = post_peak[post_peak <= threshold]
    return below.index.min() if not below.empty else pd.NaT


def eos_rows(series):
    # One EOS row per year of a smoothed NDVI series
    rows = []
    s_all = series.dropna().sort_index()
    for year, s in s_all.groupby(s_all.index.year):
        if len(s) < MIN_POINTS_PER_YEAR:
            continue
        peak = s.idxmax()
        post_peak = s[s.index >= peak]
        peak_value = s.loc[peak]
        midpoint = post_peak.min() + 0.5 * (peak_value - post_peak.min())
        eos_fixed = first_crossing(post_peak, EOS_FIXED_THRESHOLD)
        eos_mid = first_crossing(post_peak, midpoint)
        rows.append({
            "year": year,
            "n_points": len(s),
            "peak_date": peak.date(),
            "peak_ndvi": round(peak_value, 3),
            "midpoint_threshold": round(midpoint, 3),
            "eos_fixed_date": eos_fixed.date() if pd.notna(eos_fixed) else None,
            "eos_fixed_doy": eos_fixed.dayofyear if pd.notna(eos_fixed) else np.nan,
            "eos_mid_date": eos_mid.date() if pd.notna(eos_mid) else None,
            "eos_mid_doy": eos_mid.dayofyear if pd.notna(eos_mid) else np.nan,
        })
    return rows


rows = [{"zone": zone, **row} for zone in ZONES for row in eos_rows(smooth(clean_by_zone["NDVI"][zone]))]
eos_table_all = pd.DataFrame(rows).set_index(["zone", "year"])
eos_table = eos_table_all.drop(index=INCOMPLETE, level="year", errors="ignore")
eos_all = eos_table.xs("all", level="zone")
display(eos_all)

EOS_METHODS = [("eos_fixed_doy", f"EOS fixed ({EOS_FIXED_THRESHOLD})", "tab:blue"),
               ("eos_mid_doy", "EOS midpoint", "tab:red")]
fig, ax = plt.subplots(figsize=(10, 4))
for col, label, color in EOS_METHODS:
    d = eos_all[col].dropna()
    ax.plot(d.index, d.values, "o", color=color, label=label)
    if len(d) >= 3:
        fit = linregress(d.index.values, d.values)
        x = np.array([d.index.min(), d.index.max()])
        ax.plot(x, fit.intercept + fit.slope * x, "--", color=color,
                label=f"trend {fit.slope:+.2f} d/yr (p={fit.pvalue:.2f})")
ax.set_xlabel("year")
ax.set_ylabel("EOS day of year")
ax.xaxis.get_major_locator().set_params(integer=True)
ax.grid(alpha=0.3)
ax.legend()
ax.set_title("End of season per year (smoothed NDVI, whole AOI)")
plt.show()

### EOS trend test and short-term projection (no climate drivers)

- **Mann-Kendall / Theil-Sen:** a robust check for a monotonic trend. Theil-Sen gives the slope with a 95 % CI; Mann-Kendall gives the p-value.
- **Projection:** an OLS trend extrapolated `FORECAST_YEARS` ahead, with a 95 % prediction interval. A mean-only model is shown alongside. If the trend is not significant, the mean ± interval is the more honest expectation.

This is the statistical baseline that the climate models in `04` are compared against.

In [ ]:
proj_rows = []
fig, axes = plt.subplots(1, len(EOS_METHODS), figsize=(14, 4.5), sharey=True)
for ax, (col, label, color) in zip(axes, EOS_METHODS):
    d = eos_all[col].dropna()
    n = len(d)
    if n < 4:
        ax.set_title(f"{label}: only {n} years, skipped")
        continue
    years = d.index.values.astype(float)
    y = d.values.astype(float)
    ts_slope, ts_int, ts_lo, ts_hi = theilslopes(y, years, alpha=1 - ALPHA)
    tau, mk_p = kendalltau(years, y)

    x0 = years.mean()   # centred years for numerical stability
    ols = sm.OLS(y, sm.add_constant(years - x0)).fit()
    mean_model = sm.OLS(y, np.ones((n, 1))).fit()
    future = np.arange(years.max() + 1, years.max() + 1 + FORECAST_YEARS)
    grid = np.arange(years.min(), future.max() + 1)
    pred = ols.get_prediction(sm.add_constant(grid - x0, has_constant="add")).summary_frame(alpha=ALPHA)
    pred_mean = mean_model.get_prediction(np.ones((len(grid), 1))).summary_frame(alpha=ALPHA)

    print(f"{label} (n={n} years)")
    print(f"  Theil-Sen slope: {ts_slope:+.2f} d/yr  (95% CI {ts_lo:+.2f} to {ts_hi:+.2f})")
    print(f"  Mann-Kendall: tau={tau:+.2f}, p={mk_p:.3f}")
    print(f"  OLS slope: {ols.params[1]:+.2f} d/yr, p={ols.pvalues[1]:.3f}, R2={ols.rsquared:.2f}")
    for i, yr in enumerate(grid):
        if yr in future:
            proj_rows.append({
                "method": label, "year": int(yr),
                "trend_doy": pred["mean"].iloc[i],
                "trend_pi_low": pred["obs_ci_lower"].iloc[i], "trend_pi_high": pred["obs_ci_upper"].iloc[i],
                "mean_doy": pred_mean["mean"].iloc[i],
                "mean_pi_low": pred_mean["obs_ci_lower"].iloc[i], "mean_pi_high": pred_mean["obs_ci_upper"].iloc[i],
                "trend_significant": bool(mk_p < ALPHA),
            })

    ax.fill_between(grid, pred["obs_ci_lower"], pred["obs_ci_upper"], color=color, alpha=0.15,
                    label="OLS 95% prediction interval")
    ax.fill_between(grid, pred["mean_ci_lower"], pred["mean_ci_upper"], color=color, alpha=0.3,
                    label="OLS 95% CI of the trend")
    ax.plot(grid, pred["mean"], color=color, lw=2, label=f"OLS trend {ols.params[1]:+.2f} d/yr")
    ax.plot(grid, ts_int + ts_slope * grid, ":", color="black", lw=1.5,
            label=f"Theil-Sen {ts_slope:+.2f} d/yr (MK p={mk_p:.2f})")
    ax.plot(grid, pred_mean["mean"], "--", color="grey", lw=1, label="mean (no trend)")
    ax.fill_between(grid, pred_mean["obs_ci_lower"], pred_mean["obs_ci_upper"], facecolor="none",
                    edgecolor="grey", hatch="//", lw=0, alpha=0.5, label="mean 95% prediction interval")
    ax.plot(years, y, "o", color=color, label="observed EOS")
    ax.axvspan(future.min() - 0.5, future.max() + 0.5, color="grey", alpha=0.08)
    ax.set_title(label)
    ax.set_xlabel("year")
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("EOS day of year")
axes[0].legend(fontsize=8, loc="best")
plt.suptitle("EOS trend and projection (shaded columns = projected years)")
plt.tight_layout()
plt.show()

eos_projection = pd.DataFrame(proj_rows)
if not eos_projection.empty:
    eos_projection = eos_projection.set_index(["method", "year"]).round(1)
    for c in ["trend_doy", "mean_doy"]:
        eos_projection[c.replace("_doy", "_date")] = [date(int(yr), 1, 1) + timedelta(days=int(round(v)) - 1)
                                                      for (_, yr), v in eos_projection[c].items()]
    display(eos_projection)

## 4. Per-year logistic fit of the autumn transition (raw clear observations)

For each year, index and zone, a logistic curve is fitted to the raw clear observations within `FIT_WINDOW` (day of year):

`y(t) = lo + (hi - lo) / (1 + exp((t - t0) / s))`

- `hi` is the summer level, `lo` the post-transition level, `t0` the 50 % point and `s` the time scale in days.
- NDVI and GRVI drop (`hi > lo`); PSRI rises (`hi < lo`).
- **onset / mid / end** = 20 / 50 / 80 % of the transition: `t = t0 + s · ln(f / (1 − f))`.

Quality columns:
- `n_obs`: clear observations used in the fit
- `n_before_mid` / `n_after_mid`: observations on each side of `t0`
- `rmse`: fit error
- `at_bound`: a parameter sits on its fit bound. Mostly `scale_days` = 1, a step at the edge of a data gap rather than a measured transition, or `lo` = ±2, a level not constrained by any observation
- `direction_ok`: the transition goes the expected way for that index

`logistic_table` covers complete years; `logistic_table_all` also has the preliminary years. The plots show the whole AOI.

In [ ]:
def logistic(t, hi, lo, t0, s):
    return lo + (hi - lo) / (1 + np.exp((t - t0) / s))


LOGISTIC_BOUNDS = ([-2, -2, FIT_WINDOW[0], 1], [2, 2, FIT_WINDOW[1], 40])   # hi, lo, t0, s


def fit_logistic(doy, y):
    split = np.median(doy)
    p0 = [np.median(y[doy <= split]), np.median(y[doy > split]), 280, 7]
    popt, _ = curve_fit(logistic, doy, y, p0=p0, bounds=LOGISTIC_BOUNDS, maxfev=20000)
    return popt


def at_bound(popt):
    return bool(any(np.isclose(popt, b, rtol=0, atol=1e-3).any() for b in LOGISTIC_BOUNDS))


def logistic_rows(name, series, zone="all"):
    # Per-year fit of one index's clear observations; returns the table rows and {year: (doy, y, popt)}
    rows, fits = [], {}
    ser = series.dropna()
    for year, s in ser.groupby(ser.index.year):
        doy = s.index.dayofyear.values.astype(float)
        sel = (doy >= FIT_WINDOW[0]) & (doy <= FIT_WINDOW[1])
        doy, y = doy[sel], s.values[sel]
        if len(y) == 0:
            continue
        row = {"index": name, "zone": zone, "year": year, "n_obs": len(y)}
        popt = None
        if len(y) >= MIN_OBS_PER_YEAR:
            try:
                popt = fit_logistic(doy, y)
            except RuntimeError:
                print(f"{name} {zone} {year}: fit did not converge")
        if popt is not None:
            hi, lo, t0, sc = popt
            row.update({
                "hi": hi, "lo": lo, "scale_days": sc,
                "rmse": np.sqrt(np.mean((logistic(doy, *popt) - y) ** 2)),
                "n_before_mid": int((doy < t0).sum()),
                "n_after_mid": int((doy >= t0).sum()),
                "direction_ok": bool((hi > lo) == EXPECT_DECREASE[name]),
                "at_bound": at_bound(popt),
            })
            for label, f in TRANSITION_FRACTIONS.items():
                d = t0 + sc * np.log(f / (1 - f))
                row[f"{label}_doy"] = d
                row[f"{label}_date"] = date(int(year), 1, 1) + timedelta(days=int(round(d)) - 1)
        fits[year] = (doy, y, popt)
        rows.append(row)
    return rows, fits


rows, fits = [], {}
for name in INDICES:
    for zone in ZONES:
        zone_rows, zone_fits = logistic_rows(name, clean_by_zone[name][zone], zone)
        rows += zone_rows
        if zone == "all":
            fits.update({(name, year): fit for year, fit in zone_fits.items()})
logistic_table_all = pd.DataFrame(rows).set_index(["index", "zone", "year"]).round(3)
logistic_table = logistic_table_all.drop(index=INCOMPLETE, level="year", errors="ignore")
display(logistic_table_all.xs("all", level="zone"))

fit_years = sorted({yr for _, yr in fits})
tt = np.linspace(*FIT_WINDOW, 300)
fig, axes = plt.subplots(len(INDICES), len(fit_years), squeeze=False, sharex=True,
                         figsize=(2.4 * len(fit_years), 2.4 * len(INDICES)))
for i, name in enumerate(INDICES):
    for j, year in enumerate(fit_years):
        ax = axes[i, j]
        if (name, year) in fits:
            doy, y, popt = fits[(name, year)]
            ax.plot(doy, y, "o", ms=3, color=INDEX_COLORS[name])
            if popt is not None:
                ax.plot(tt, logistic(tt, *popt), color="black", lw=1)
                for label, f in TRANSITION_FRACTIONS.items():
                    ax.axvline(popt[2] + popt[3] * np.log(f / (1 - f)), color="red",
                               lw=1.2 if label == "mid" else 0.6, ls="--")
        if i == 0:
            ax.set_title(f"{year} (preliminary)" if year in INCOMPLETE else str(year), fontsize=9)
        if j == 0:
            ax.set_ylabel(name)
        if i == len(INDICES) - 1:
            ax.set_xlabel("DOY")
        ax.grid(alpha=0.3)
plt.suptitle("Per-year logistic fit of the autumn transition (red dashed: onset / mid / end)")
plt.tight_layout()
plt.show()

## 5. Season dates with standard methods

The methods below follow the operational land-surface-phenology products and the literature (overview: Zeng et al. 2020). All of them work on the clear observations themselves, not on a moving average, and all report dates at fixed shares of the autumn amplitude (`AMPLITUDE_FRACTIONS`).

| Method | Curve | Dates | Source |
|---|---|---|---|
| `logistic, fixed base` | autumn logistic, post-season level fixed | 90 / 50 / 15 % | Beck et al. 2006 (fixed base); MCD12Q2, HR-VPP (thresholds) |
| `double logistic` | full-season double logistic, base fixed | 90 / 50 / 15 % | Beck et al. 2006; TIMESAT / HR-VPP (Jönsson & Eklundh 2004) |
| `double logistic, 3rd derivative` | same fit | zeros of the 3rd derivative | Busetto et al. 2010 (larch, Alps) |
| `Whittaker` | weighted Whittaker smoother | 90 / 50 / 15 %, and the yellowness `peak` | Eilers 2003; Atzberger & Eilers 2011 |
| `logistic, free` | section 4 | 80 / 50 / 20 % (`TRANSITION_FRACTIONS`) | earlier version |
| `MA midpoint`, `MA fixed threshold` | section 3 (NDVI only) | moving-average thresholds (`EOS_FIXED_THRESHOLD`) | earlier version |

### 5a. Observations, weights, outliers and the post-season level

- **Weights** follow the HR-VPP quality classes: 1 for a good date (clear share ≥ `GOOD_VALID_FRACTION`), 0.5 for a usable one, 0 for a date with snow on more than `MAX_SNOW_FRACTION` of the zone.
- **Outliers** follow the HR-VPP rule: a date is dropped (weight 0) if its residual from a stiff smoothing curve exceeds `OUTLIER_FACTOR` × the global and × the local (91-day) median residual.
- **Post-season level ("base"):** the snow-free level after needle fall, the median of the clear dates in `POST_WINDOW` over all years. It is held fixed in the fits instead of being fitted per year, as Beck et al. (2006) do with the winter NDVI. Without a fixed base, a fit with no data after the decline puts the lower level anywhere (the `at_bound` fits of section 4). Zones with fewer than `MIN_BASE_OBS` such dates use the whole AOI's level.

In [ ]:
METHOD_COLORS = {"MA midpoint": "tab:purple", "MA fixed threshold": "plum", "logistic, free": "tab:red",
                 "logistic, fixed base": "tab:blue", "double logistic": "tab:orange",
                 "double logistic, 3rd derivative": "tab:brown", "Whittaker": "tab:green"}
METRIC_STYLES = {"start": ":", "mid": "-", "end": "--", "peak": "-."}


def obs_frame(name, zone):
    # Clear observations of one index and zone: y (zone median), w (weight), year, doy. Weights as the HR-VPP
    # quality classes (ATBD; Tian et al. 2021): 1 = good, 0.5 = usable, 0 = snow (kept for the plots, not fitted)
    f = clean_frame(stats_all, name.lower(), zone)
    obs = pd.DataFrame({"y": f[f"{name.lower()}_median"],
                        "w": np.where(f.valid_fraction >= GOOD_VALID_FRACTION, 1.0, 0.5)}, index=f.index)
    obs.loc[f.scl_snow_fraction > MAX_SNOW_FRACTION, "w"] = 0.0
    obs = obs.dropna(subset=["y"])
    return obs.assign(year=obs.index.year, doy=obs.index.dayofyear, outlier=False)


def base_levels(name):
    # Post-season level per zone, held fixed in all fits (Beck et al. 2006): median of the weighted dates in
    # POST_WINDOW over all years. Fallbacks: the whole AOI's level; then, as HR-VPP sets its base level, the 5th
    # percentile of all clear dates (95th for rising indices, whose post-season level is the high one)
    rows = {}
    for zone in ZONES:
        o = obs_by_zone[name][zone]
        post = o[o.doy.between(*POST_WINDOW) & (o.w > 0)]
        rows[zone] = {"level": post.y.median(), "n_dates": len(post), "source": "zone"}
    for zone, r in rows.items():
        if r["n_dates"] < MIN_BASE_OBS and rows["all"]["n_dates"] >= MIN_BASE_OBS:
            r.update(level=rows["all"]["level"], source="whole AOI")
        elif r["n_dates"] < MIN_BASE_OBS:
            q = 0.05 if EXPECT_DECREASE[name] else 0.95
            r.update(level=obs_by_zone[name][zone].y.quantile(q), source=f"{q:.0%} percentile")
    return pd.DataFrame(rows).T.astype({"level": float, "n_dates": int})


# ---- Weighted Whittaker smoother (also used for the outlier test, so it is defined here)
def whittaker(days, y, w, lam, n):
    # Eilers (2003), for vegetation indices Atzberger & Eilers (2011): on a daily grid of n days, minimise
    # sum w (y - z)^2 + lam * sum (2nd difference of z)^2. Days without an observation have weight 0, so irregular
    # dates and gaps need no interpolation beforehand; across a gap the curve becomes nearly a straight line.
    W, Y = np.zeros(n), np.zeros(n)
    np.add.at(W, days, w)
    np.add.at(Y, days, w * y)
    D = sparse.diags([1.0, -2.0, 1.0], [0, 1, 2], shape=(n - 2, n))
    return spsolve((sparse.diags(W) + lam * (D.T @ D)).tocsc(), Y)


def smooth_series(obs, lam, base):
    # Daily Whittaker curve (pd.Series over whole calendar years) through the weighted dates of obs. Long
    # off-season gaps are filled as in HR-VPP: if more than 90 days pass without a good date between the autumn
    # and the spring equinox, control points at the base level (weight 0.5) are added every 15 days.
    o = obs[obs.w > 0]
    start = pd.Timestamp(obs.index.min().year, 1, 1)
    end = pd.Timestamp(obs.index.max().year, 12, 31)
    grid = pd.date_range(start, end)
    days, y, w = (o.index - start).days.values, o.y.values, o.w.values
    control = [d for a, b in zip(days[:-1], days[1:]) if b - a > 90 for d in range(a + 15, b - 7, 15)
               if not 80 < grid[d].dayofyear < 266]
    days = np.concatenate([days, control]).astype(int)
    y = np.concatenate([y, np.full(len(control), base)])
    w = np.concatenate([w, np.full(len(control), 0.5)])
    return pd.Series(whittaker(days, y, w, lam, len(grid)), index=grid)


def flag_outliers(obs, base):
    # HR-VPP outlier rule (ATBD 4.2.1): residuals from a stiff curve; outlier if the residual exceeds
    # OUTLIER_FACTOR x the global median residual and OUTLIER_FACTOR x the median of the surrounding 91 days
    o = obs[obs.w > 0]
    curve = smooth_series(obs, max(WHITTAKER_LAMBDAS), base)
    res = (o.y - curve.reindex(o.index).values).abs()
    local = res.rolling("91D", center=True, min_periods=1).median()
    out = res.index[(res > OUTLIER_FACTOR * res.median()) & (res > OUTLIER_FACTOR * local)]
    obs = obs.copy()
    obs.loc[out, ["w", "outlier"]] = [0.0, True]
    return obs


obs_by_zone = {name: {zone: obs_frame(name, zone) for zone in ZONES} for name in INDICES}
base = {name: base_levels(name) for name in INDICES}
for name in INDICES:
    for zone in ZONES:
        obs_by_zone[name][zone] = flag_outliers(obs_by_zone[name][zone], base[name].level[zone])

display(pd.concat({name: b for name, b in base.items()}, axis=1).round(3))
print("Dates per index and zone: weight 1 / 0.5 / 0 (snow) / outliers")
display(pd.DataFrame({name: {zone: f"{(o.w == 1).sum()} / {(o.w == 0.5).sum()} / {((o.w == 0) & ~o.outlier).sum()} / "
                                   f"{o.outlier.sum()}" for zone, o in by_zone.items()}
                      for name, by_zone in obs_by_zone.items()}))

### 5b. Whittaker smoother

The weighted Whittaker smoother (Eilers 2003) fits a smooth daily curve to the irregular observations; Atzberger & Eilers (2011) introduced it for vegetation-index time series. It needs no gap interpolation beforehand, and one parameter, λ, sets the smoothness. λ is chosen per index from `WHITTAKER_LAMBDAS` by 10-fold cross-validation on the whole AOI (weighted RMSE of left-out dates), then used for every zone. HR-VPP assumes clear Sentinel-2 dates are unbiased and applies no upper-envelope correction; neither does this notebook.

In [ ]:
def choose_lambda(obs, base_level, k=10, seed=0):
    # Weighted RMSE of left-out good dates for every lam (k-fold cross-validation)
    good = obs[obs.w > 0]
    fold = np.random.default_rng(seed).integers(0, k, len(good))
    rmse = {}
    for lam in WHITTAKER_LAMBDAS:
        sq = []
        for i in range(k):
            test = good[fold == i]
            curve = smooth_series(obs.drop(index=test.index), lam, base_level)
            sq.append(test.w * (test.y - curve.reindex(test.index).values) ** 2)
        rmse[lam] = np.sqrt(pd.concat(sq).sum() / good.w.sum())
    return pd.Series(rmse)


cv_lambda = pd.DataFrame({name: choose_lambda(obs_by_zone[name]["all"], base[name].level["all"]) for name in INDICES})
LAMBDA = cv_lambda.idxmin()
print("Cross-validated RMSE per lambda (whole AOI); chosen:", LAMBDA.to_dict())
display(cv_lambda.rename_axis("lambda").round(4))
wcurves = {name: {zone: smooth_series(obs_by_zone[name][zone], LAMBDA[name], base[name].level[zone]) for zone in ZONES}
           for name in INDICES}

### 5c. Logistic fits with a fixed post-season level

- **`logistic, fixed base`:** the autumn logistic of section 4, `y = lo + (hi − lo) / (1 + exp((t − t0) / s))`, but with `lo` fixed at the post-season level (5a) and the dates weighted. Only `hi`, `t0` and `s` are fitted, to the weighted dates in `FIT_WINDOW[0]`–`POST_WINDOW[1]`.
- **`double logistic`:** the function of Beck et al. (2006) over the whole snow-free season (`SEASON_WINDOW`), `y = b + (m − b)·(1/(1 + exp(−mS(t − S))) + 1/(1 + exp(mA(t − A))) − 1)`, with the base `b` fixed. It is the function type TIMESAT fits for HR-VPP; only its autumn half (`A`, `mA`) gives dates here.
- **Dates:** a share `f` of the amplitude is left at `t = t0 + s · ln((1 − f) / f)` (for the double logistic `t0 = A`, `s = 1 / mA`).
- **`double logistic, 3rd derivative`:** Busetto et al. (2010) found, for larch in the Aosta Valley, that the zeros of the third derivative of a double-logistic NDVI curve matched field observations best (MAE 4 days for the end). For a logistic they lie at `t0 ∓ s · ln(2 + √3)`, where about 79 % and 21 % of the amplitude are left.
- **Uncertainty:** the fixed-base logistic is refitted to `N_BOOT` bootstrap resamples of each year's dates; `ci_low` / `ci_high` are the 2.5 / 97.5 % quantiles of each date.
- **`at_bound`:** `t0` (or `A`) or the time scale sits on its bound.

In [ ]:
TURNING = np.log(2 + np.sqrt(3))   # 3rd-derivative zeros of a logistic: t0 -/+ 1.317 s (Busetto et al. 2010)
QA_LEVELS = ["filled", "low", "medium", "high"]


def logistic_base(t, hi, t0, s, lo):
    return lo + (hi - lo) / (1 + np.exp((t - t0) / s))


def double_logistic(t, m, S, mS, A, mA, b):
    # Beck et al. (2006): b = off-season level (fixed), m = summer level, S / A = spring / autumn inflection day,
    # mS / mA = their rates (1/day)
    return b + (m - b) * (1 / (1 + np.exp(-mS * (t - S))) + 1 / (1 + np.exp(mA * (t - A))) - 1)


def fraction_day(t0, s, f):
    # Day on a logistic when a share f of the amplitude is left
    return t0 + s * np.log((1 - f) / f)


def fit_fixed_base(doy, y, w, lo):
    p0 = [np.clip(np.median(y[doy <= np.median(doy)]), -1.9, 1.9), 280, 7]
    bounds = ([-2, FIT_WINDOW[0], 1], [2, POST_WINDOW[1], 40])
    popt, _ = curve_fit(lambda t, hi, t0, s: logistic_base(t, hi, t0, s, lo), doy, y, p0=p0,
                        sigma=1 / np.sqrt(w), bounds=bounds, maxfev=20000)
    return popt


def fit_double_logistic(doy, y, w, b):
    summer = y[(doy >= SUMMER_WINDOW[0]) & (doy <= SUMMER_WINDOW[1])]
    p0 = [np.clip(np.median(summer if len(summer) else y), -1.9, 1.9), 160, 0.1, 280, 0.1]
    bounds = ([-2, SEASON_WINDOW[0], 1 / 40, SUMMER_WINDOW[0], 1 / 40], [2, SUMMER_WINDOW[0], 1, SEASON_WINDOW[1], 1])
    popt, _ = curve_fit(lambda t, m, S, mS, A, mA: double_logistic(t, m, S, mS, A, mA, b), doy, y, p0=p0,
                        sigma=1 / np.sqrt(w), bounds=bounds, maxfev=20000)
    return popt


def hrvpp_qa(doy, t80, t20):
    # Adapted from the HR-VPP QA (ATBD Table 3): clear dates are counted in three parts of the curve - plateau
    # (30 days before 80 % of the amplitude is left), decline (80 % -> 20 %) and after (30 days past 20 %).
    # high = more than 2 dates in all three parts, medium = in two, low = in one, filled = in none
    parts = [(doy >= t80 - 30) & (doy < t80), (doy >= t80) & (doy <= t20), (doy > t20) & (doy <= t20 + 30)]
    return QA_LEVELS[sum(int(p.sum()) > 2 for p in parts)]


def on_bound(value, low, high):
    return bool(np.isclose(value, low, atol=1e-3) or np.isclose(value, high, atol=1e-3))


rng = np.random.default_rng(0)
rows, fixed_fits, dl_fits = [], {}, {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore", OptimizeWarning)
    for name in INDICES:
        for zone in ZONES:
            obs, lo = obs_by_zone[name][zone], base[name].level[zone]
            for year, o in obs[obs.w > 0].groupby("year"):
                key = dict(index=name, zone=zone, year=year)
                autumn = o[o.doy.between(FIT_WINDOW[0], POST_WINDOW[1])]
                if len(autumn) < MIN_OBS_PER_YEAR:
                    continue
                doy, y, w = autumn.doy.values.astype(float), autumn.y.values, autumn.w.values
                # -- fixed-base logistic + bootstrap
                try:
                    hi, t0, s = fit_fixed_base(doy, y, w, lo)
                except RuntimeError:
                    hi = None
                if hi is not None:
                    fixed_fits[(name, zone, year)] = (hi, t0, s, lo)
                    boot = []
                    for _ in range(N_BOOT):
                        i = rng.integers(0, len(doy), len(doy))
                        try:
                            b = fit_fixed_base(doy[i], y[i], w[i], lo)
                        except (RuntimeError, ValueError):
                            continue
                        boot.append([fraction_day(b[1], b[2], f) for f in AMPLITUDE_FRACTIONS.values()])
                    boot = np.array(boot)
                    qa = hrvpp_qa(doy, fraction_day(t0, s, 0.8), fraction_day(t0, s, 0.2))
                    bound = on_bound(t0, FIT_WINDOW[0], POST_WINDOW[1]) or on_bound(s, 1, 40)
                    for k, (metric, f) in enumerate(AMPLITUDE_FRACTIONS.items()):
                        ci = (np.percentile(boot[:, k], [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)])
                              if len(boot) >= 20 else [np.nan, np.nan])
                        rows.append({**key, "method": "logistic, fixed base", "metric": metric,
                                     "doy": fraction_day(t0, s, f), "ci_low": ci[0], "ci_high": ci[1], "qa": qa,
                                     "at_bound": bound, "n_obs": len(doy)})
                # -- double logistic over the season
                season = o[o.doy.between(*SEASON_WINDOW)]
                try:
                    m, S, mS, A, mA = fit_double_logistic(season.doy.values.astype(float), season.y.values,
                                                          season.w.values, lo)
                except RuntimeError:
                    continue
                dl_fits[(name, zone, year)] = (m, S, mS, A, mA, lo)
                qa = hrvpp_qa(doy, fraction_day(A, 1 / mA, 0.8), fraction_day(A, 1 / mA, 0.2))
                bound = on_bound(A, SUMMER_WINDOW[0], SEASON_WINDOW[1]) or on_bound(mA, 1 / 40, 1)
                common = {**key, "qa": qa, "at_bound": bound, "n_obs": len(season)}
                for metric, f in AMPLITUDE_FRACTIONS.items():
                    rows.append({**common, "method": "double logistic", "metric": metric,
                                 "doy": fraction_day(A, 1 / mA, f)})
                for metric, sign in [("start", -1), ("end", 1)]:
                    rows.append({**common, "method": "double logistic, 3rd derivative", "metric": metric,
                                 "doy": A + sign * TURNING / mA})
fit_rows = rows
print(f"{len(fixed_fits)} fixed-base logistic and {len(dl_fits)} double-logistic fits")

### 5d. Dates from the Whittaker curve and the yellowing peak

- **Start / mid / end:** per year, the summer level `hi` is the curve's median in `SUMMER_WINDOW`; the base is the post-season level. With `r` = share of the amplitude left:
  - mid = first day after the summer window with `r ≤ 0.5`
  - start = last day before mid with `r ≥ 0.9` (MCD12Q2: "last date above 90 %")
  - end = first day after mid with `r ≤ 0.15`

  A date that is not reached within the year stays empty.
- **Peak of yellowing:** the day of the curve's maximum in `PEAK_WINDOW` for the yellowness indices of `PEAK_INDICES` (RCC and NDYI, once downloaded, and PSRI). PhenoCam studies track autumn colour with the chromatic coordinates (Sonnentag et al. 2012; Richardson et al. 2018). NDYI was built for yellow canopies (Sulik & Long 2016). PSRI rises with senescence (Merzlyak et al. 1999), but litter and soil keep it high after needle fall, so its maximum may come late. `at_bound` = maximum on the edge of the window. The QA counts clear dates within ±10 days: high ≥ 3, medium 2, low 1, filled 0.

In [ ]:
def whittaker_dates(curve, hi, lo):
    # Start / mid / end on one year of a daily curve (index = DOY); also the 80 % / 20 % days for the QA
    r = (curve - lo) / (hi - lo)
    below_mid = r[(r.index > SUMMER_WINDOW[1]) & (r <= AMPLITUDE_FRACTIONS["mid"])]
    if below_mid.empty:
        return None
    mid = below_mid.index[0]

    def last_above(f):
        above = r[(r.index < mid) & (r >= f)]
        return above.index[-1] if not above.empty else np.nan

    def first_below(f):
        below = r[(r.index > mid) & (r <= f)]
        return below.index[0] if not below.empty else np.nan

    days = {"start": last_above(AMPLITUDE_FRACTIONS["start"]), "mid": mid, "end": first_below(AMPLITUDE_FRACTIONS["end"])}
    return days, last_above(0.8), first_below(0.2)


rows = []
for name in INDICES:
    for zone in ZONES:
        obs, lo = obs_by_zone[name][zone], base[name].level[zone]
        curve_all = wcurves[name][zone]
        for year in sorted(obs.year.unique()):
            curve = curve_all[curve_all.index.year == year]
            curve.index = curve.index.dayofyear
            doy = obs[(obs.year == year) & (obs.w > 0)].doy.values
            key = dict(index=name, zone=zone, year=year, method="Whittaker")
            hi = curve.loc[SUMMER_WINDOW[0]:SUMMER_WINDOW[1]].median()
            found = whittaker_dates(curve, hi, lo) if (hi > lo) == EXPECT_DECREASE[name] else None
            if found:
                days, t80, t20 = found
                qa = hrvpp_qa(doy, t80, t20) if np.isfinite([t80, t20]).all() else "filled"
                rows += [{**key, "metric": m, "doy": d, "qa": qa, "at_bound": False, "n_obs": len(doy)}
                         for m, d in days.items()]
            if name in PEAK_INDICES:
                window = curve.loc[PEAK_WINDOW[0]:PEAK_WINDOW[1]]
                peak = window.idxmax()
                near = int((np.abs(doy - peak) <= 10).sum())
                rows.append({**key, "metric": "peak", "doy": peak, "qa": QA_LEVELS[min(near, 3)],
                             "at_bound": peak in (window.index[0], window.index[-1]), "n_obs": len(doy)})
whittaker_rows = rows
print(f"{len(whittaker_rows)} Whittaker dates")

### 5e. All dates in one table

`season_dates` holds every date of every method, index, zone and year (long format), including the earlier methods of sections 3 and 4:

| Column | Meaning |
|---|---|
| `metric` | start, mid, peak or end |
| `doy`, `date` | the estimated date |
| `ci_low`, `ci_high` | bootstrap 95 % interval (fixed-base logistic only) |
| `qa` | HR-VPP-style quality (curve methods): high / medium / low / filled |
| `at_bound` | fit parameter on its bound |
| `bracket_days` | days between the clear dates just before and after the estimate; more than `GAP_DAYS` = the date lies inside a data gap |
| `trusted` | `bracket_days` ≤ `GAP_DAYS`, `qa` not low / filled and not `at_bound` |
| `preliminary` | year in `INCOMPLETE` |

In [ ]:
rows = []
for (zone, year), r in eos_table_all.iterrows():
    rows.append({"index": "NDVI", "zone": zone, "year": year, "method": "MA midpoint", "metric": "mid", "doy": r.eos_mid_doy})
    rows.append({"index": "NDVI", "zone": zone, "year": year, "method": "MA fixed threshold", "metric": "end",
                 "doy": r.eos_fixed_doy})
for (name, zone, year), r in logistic_table_all.iterrows():
    if pd.isna(r.mid_doy):
        continue
    o = obs_by_zone[name][zone]
    doy = o[(o.year == year) & o.doy.between(*FIT_WINDOW)].doy.values
    qa = hrvpp_qa(doy, r.onset_doy, r.end_doy)
    rows += [{"index": name, "zone": zone, "year": year, "method": "logistic, free", "metric": m, "doy": r[f"{c}_doy"],
              "qa": qa, "at_bound": bool(r.at_bound), "n_obs": r.n_obs}
             for m, c in [("start", "onset"), ("mid", "mid"), ("end", "end")]]

season_dates = pd.DataFrame(rows + fit_rows + whittaker_rows)
season_dates = season_dates[season_dates.doy.notna()].reset_index(drop=True)


def bracket_days(group):
    # Days between the last clear date before and the first after each estimate (inf if none on a side)
    name, zone, year = group.name
    o = obs_by_zone[name][zone]
    doy = np.concatenate([[-np.inf], np.sort(o[(o.year == year) & (o.w > 0)].doy.values.astype(float)), [np.inf]])
    i = np.searchsorted(doy, group.doy.values, side="right")
    return pd.Series(doy[i] - doy[i - 1], index=group.index)


season_dates["bracket_days"] = season_dates.groupby(["index", "zone", "year"], group_keys=False)[["doy"]].apply(
    bracket_days)
season_dates["trusted"] = ((season_dates.bracket_days <= GAP_DAYS) & ~season_dates.qa.isin(["low", "filled"])
                           & ~season_dates.at_bound.eq(True))
season_dates["preliminary"] = season_dates.year.isin(INCOMPLETE)
season_dates["date"] = pd.to_datetime(season_dates.year.astype(str)) + pd.to_timedelta(season_dates.doy.round() - 1, unit="D")
season_dates = season_dates.sort_values(["metric", "method", "index", "zone", "year"]).reset_index(drop=True)

print("Whole AOI, complete years: day of year per method and index")
aoi = season_dates[(season_dates.zone == "all") & ~season_dates.preliminary]
display(aoi.pivot_table(index=["metric", "method", "index"], columns="year", values="doy").round(0))
print("Share of trusted dates (all zones, complete years)")
display(season_dates[~season_dates.preliminary].pivot_table(index=["method", "index"], columns="metric",
                                                             values="trusted", aggfunc="mean").round(2))

## 6. Diagnostic panel per year

One figure per zone in `PANEL_ZONES`, one row per year, `PANEL_WINDOW` on the x axis. It shows which dates follow the observations and which only mark the edge of a data gap.

- **Index columns:**
  - dots = clear dates (grey: failed the filter; ×: weight 0, i.e. snow or outlier)
  - black line = Whittaker curve; light grey line = moving average; dotted horizontal = post-season level
  - thin coloured curves = the fitted logistic / double logistic of the year (if in `PANEL_METHODS`)
  - vertical lines = dates of the methods in `PANEL_METHODS` (colour = method; dotted start, solid mid, dash-dot peak, dashed end), faint if not `trusted`
  - shaded = more than `GAP_DAYS` days without a clear date
- **Data column:** share of the zone's pixels per date that are clear (bars; dotted line = `MIN_VALID_FRACTION`), snow, SCL dark and cloud.

The table below lists the clear dates per half month and the last clear date of each autumn.

In [ ]:
SHARE_STYLES = {"scl_snow_fraction": ("snow", "tab:cyan"), "scl_dark_fraction": ("SCL dark", "black"),
                "scl_cloud_fraction": ("cloud", "darkgrey")}
MONTH_TICKS = {date(2023, m, 1).timetuple().tm_yday: f"1 {date(2023, m, 1):%b}" for m in range(8, 13)}
FITTED_CURVES = {"logistic, fixed base": (fixed_fits, logistic_base), "double logistic": (dl_fits, double_logistic)}


def in_window(frame, year):
    doy = frame.index.dayofyear
    return frame[(frame.index.year == year) & (doy >= PANEL_WINDOW[0]) & (doy <= PANEL_WINDOW[1])]


def clear_gaps(doy):
    # (from, to) DOY spans longer than GAP_DAYS without a clear observation within PANEL_WINDOW
    edges = np.concatenate([[PANEL_WINDOW[0]], np.sort(doy), [PANEL_WINDOW[1]]])
    return [(a, b) for a, b in zip(edges[:-1], edges[1:]) if b - a > GAP_DAYS]


def legend_handles():
    line = lambda **kw: plt.Line2D([], [], **kw)
    return [line(marker="o", ls="", color="lightgrey", label="filtered out"),
            line(marker="o", ls="", color="darkgreen", label="clear date (colour per index)"),
            line(marker="x", ls="", color="darkgreen", label="weight 0 (snow / outlier)"),
            line(color="black", label="Whittaker curve"),
            line(color="lightgrey", label=f"{SMOOTH_WINDOW * 5}-day moving average"),
            line(color="grey", ls=":", label="post-season level"),
            *[line(color=METHOD_COLORS[m], lw=2, label=m) for m in PANEL_METHODS],
            *[line(color="black", ls=ls, label=k) for k, ls in METRIC_STYLES.items()],
            plt.Rectangle((0, 0), 1, 1, color="grey", alpha=0.15, label=f"> {GAP_DAYS} days without clear date"),
            plt.Rectangle((0, 0), 1, 1, color="tab:green", alpha=0.5, label="clear share"),
            *[line(color=c, marker=".", label=f"{n} share") for n, c in SHARE_STYLES.values()]]


def year_panel(zone):
    years = sorted(set(stats_all.index.year))
    ncol = len(INDICES) + 1
    fig, axes = plt.subplots(len(years), ncol, figsize=(3.3 * ncol, 1.8 * len(years) + 1.2), sharex=True,
                             sharey="col", squeeze=False)
    dates = season_dates[(season_dates.zone == zone) & season_dates.method.isin(PANEL_METHODS)]
    tt = np.arange(PANEL_WINDOW[0], PANEL_WINDOW[1] + 1)
    for j, name in enumerate(INDICES):
        everything = index_frame(stats_all, name.lower(), zone)[f"{name.lower()}_median"]
        moving = smooth(clean_by_zone[name][zone])
        for i, year in enumerate(years):
            ax = axes[i, j]
            o = in_window(obs_by_zone[name][zone], year)
            for g0, g1 in clear_gaps(o[o.w > 0].doy.values):
                ax.axvspan(g0, g1, color="grey", alpha=0.15, lw=0)
            a = in_window(everything, year)
            ax.scatter(a.index.dayofyear, a, s=6, color="lightgrey")
            ax.scatter(o[o.w > 0].doy, o[o.w > 0].y, s=12, color=INDEX_COLORS[name])
            ax.scatter(o[o.w == 0].doy, o[o.w == 0].y, s=16, marker="x", color=INDEX_COLORS[name])
            m = in_window(moving, year)
            ax.plot(m.index.dayofyear, m, color="lightgrey", lw=1)
            z = in_window(wcurves[name][zone], year)
            ax.plot(z.index.dayofyear, z, color="black", lw=1)
            ax.axhline(base[name].level[zone], color="grey", ls=":", lw=0.8)
            for method, (fits_, f) in FITTED_CURVES.items():
                if method in PANEL_METHODS and (name, zone, year) in fits_:
                    ax.plot(tt, f(tt, *fits_[(name, zone, year)]), color=METHOD_COLORS[method], lw=0.8)
            for r in dates[(dates["index"] == name) & (dates.year == year)].itertuples():
                ax.axvline(r.doy, color=METHOD_COLORS[r.method], ls=METRIC_STYLES[r.metric], lw=1.3,
                           alpha=1 if r.trusted else 0.3)
            ax.grid(alpha=0.3)
    for i, year in enumerate(years):
        ax = axes[i, -1]
        q = in_window(index_frame(stats_all, "ndvi", zone), year)
        ax.bar(q.index.dayofyear, q.valid_fraction, width=3, color="tab:green", alpha=0.5)
        for col, (_, color) in SHARE_STYLES.items():
            ax.plot(q.index.dayofyear, q[col], ".-", color=color, lw=0.6, ms=3)
        ax.axhline(MIN_VALID_FRACTION, color="tab:green", ls=":", lw=1)
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.3)
        axes[i, 0].set_ylabel(f"{year}\n(preliminary)" if year in INCOMPLETE else str(year))
    for j, title in enumerate(INDICES + ["data (share of zone pixels)"]):
        axes[0, j].set_title(title)
        axes[-1, j].set_xticks(list(MONTH_TICKS), list(MONTH_TICKS.values()))
    axes[0, 0].set_xlim(*PANEL_WINDOW)
    height = fig.get_size_inches()[1]
    fig.legend(handles=legend_handles(), loc="upper center", ncol=8, fontsize=8, bbox_to_anchor=(0.5, 1))
    fig.suptitle(f"Zone {zone}: observations and season dates per year", y=1 - 0.75 / height)
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.95 / height))
    plt.show()


def coverage_table(zone):
    # Clear dates per half month (Sep-Nov) and the last clear date of each autumn
    c = clean_by_zone["NDVI"][zone]
    c = c[c.index.month.isin([9, 10, 11])]
    half = c.index.strftime("%b") + np.where(c.index.day <= 15, " 1-15", " 16-")
    t = pd.crosstab(c.index.year, half).reindex(columns=[f"{m} {h}" for m in ["Sep", "Oct", "Nov"]
                                                         for h in ["1-15", "16-"]], fill_value=0)
    t["last_clear"] = c.groupby(c.index.year).apply(lambda s: s.index.max().strftime("%d %b"))
    t["preliminary"] = t.index.isin(INCOMPLETE)
    return t.rename_axis(index="year", columns=None)


for zone in [z for z in PANEL_ZONES if z in ZONES]:
    year_panel(zone)
    print(f"Clear dates per half month, zone {zone}")
    display(coverage_table(zone))

## 7. Comparison of all methods

One table and one figure over every candidate (method × index × date type). Peak candidates are the yellowness maxima (`peak`) and the 50 % points of the declines (`mid`, the MCD12Q2 "greendown midpoint"). Complete years only.

| Column | Meaning |
|---|---|
| `coverage` | share of zone-years with a date |
| `observed` | share of dates with a clear date within `GAP_DAYS` on both sides (not inside a data gap) |
| `qa_good` | share with QA high or medium (curve methods) |
| `ci_days` | median width of the bootstrap 95 % interval (fixed-base logistic) |
| `order_ok` | share of zone-years in the order start < peak < end, judged against the median of all candidates of the neighbouring date types |
| `days_per_100m` | within-year elevation gradient across zones; negative = earlier higher up, as expected (elevation controls larch phenology; Migliavacca et al. 2008) |
| `aoi_sd_years` | year-to-year standard deviation of the whole-AOI date |
| `ms_r`, `ms_rmse`, `ms_n` | whole-AOI anomalies vs MeteoSwiss station anomalies: needle colouring 50 % for the peak, needle drop 50 % for the end. Correlation, RMSE in days, number of years |
| `ms_offset` | mean AOI date − mean station date. The stations lie lower (1050–1930 m), so this is not an error |
| `hrvpp_mae`, `hrvpp_bias`, `hrvpp_r` | end dates vs HR-VPP end of season per zone-year (if downloaded in `01b`) |

**Short list:** `mean_rank` is the mean of a candidate's ranks within its date type over `ms_r`, `hrvpp_mae`, `observed`, `order_ok` and `qa_good` (where available; rank 1 = best). The three best candidates with `coverage` ≥ 0.5 are printed. This is a heuristic list to check in the panel, not a final choice. With about 9 years, a correlation needs |r| > 0.67 to be significant at 5 %.

In [ ]:
TARGET_METRICS = {"start": ["start"], "peak": ["peak", "mid"], "end": ["end"]}
VALIDATE_WITH = {"peak": "mlard94d", "end": "mlard95d"}      # MeteoSwiss phase per date type
d = season_dates[~season_dates.preliminary].dropna(subset=["doy"])
complete_years = [y for y in sorted(set(stats_all.index.year)) if y not in INCOMPLETE]
n_zone_years = (len(ZONES) - 1) * len(complete_years)

# ---- validation data from 01b_download_validation (each part is skipped if its file is missing)
pheno, hrvpp_end = None, None
if (VALIDATION_DIR / "meteoswiss_larch.csv").exists():
    p = pd.read_csv(VALIDATION_DIR / "meteoswiss_larch.csv")
    p = p[p.year.isin(complete_years)]
    p["anomaly"] = p.doy - p.groupby(["station", "param"]).doy.transform("mean")   # each station vs its own mean
    pheno = p.groupby(["param", "year"]).agg(anomaly=("anomaly", "mean"), doy=("doy", "mean"),
                                            n_stations=("station", "nunique"), height=("height_masl", "mean"))
else:
    print("No MeteoSwiss observations (run 01b_download_validation)")
if (VALIDATION_DIR / "hrvpp_by_zone.csv").exists():
    h = pd.read_csv(VALIDATION_DIR / "hrvpp_by_zone.csv")
    hrvpp_end = h[h["product"] == "EOSD"].set_index(["zone", "year"]).doy_median.dropna()
if hrvpp_end is None or hrvpp_end.empty:
    hrvpp_end = None
    print("No HR-VPP end-of-season dates (needs a WEkEO login, see 01b_download_validation)")

# ---- candidates and the consensus (median of all candidates) per zone-year and date type
candidates = {(t, *k): g.set_index(["zone", "year"]).doy for t, metrics in TARGET_METRICS.items()
              for k, g in d[d.metric.isin(metrics)].groupby(["method", "index", "metric"])}
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # zone-years without any candidate of a date type
    consensus = pd.DataFrame({t: pd.concat([s for k, s in candidates.items() if k[0] == t], axis=1).median(axis=1)
                              for t in TARGET_METRICS})


def order_share(target, s):
    c = consensus.reindex(s.index)
    ok = {"start": s < c["peak"], "peak": (c["start"] < s) & (s < c["end"]), "end": s > c["peak"]}[target]
    known = {"start": c["peak"].notna(), "peak": c["start"].notna() & c["end"].notna(), "end": c["peak"].notna()}[target]
    return ok[known].mean() if known.any() else np.nan


def elevation_slope(s):
    # Days per 100 m from zone dates, within years (each year's mean removed from date and elevation)
    z = s.drop(index="all", level="zone", errors="ignore").reset_index()
    z["x"] = z.zone.map(zone_table.elev_mean) / 100
    z = z[z.groupby("year").x.transform("size") >= 2]
    x, y = z.x - z.groupby("year").x.transform("mean"), z.doy - z.groupby("year").doy.transform("mean")
    return (x @ y) / (x @ x) if len(z) > 3 and (x @ x) > 0 else np.nan


def meteoswiss_check(target, aoi):
    if pheno is None or target not in VALIDATE_WITH:
        return {}
    st = pheno.xs(VALIDATE_WITH[target], level="param")
    common = aoi.index.intersection(st.index)
    if len(common) < 4:
        return {"ms_n": len(common)}
    a = aoi[common] - aoi[common].mean()
    s = st.anomaly[common] - st.anomaly[common].mean()
    return {"ms_r": a.corr(s), "ms_rmse": np.sqrt(((a - s) ** 2).mean()), "ms_n": len(common),
            "ms_offset": (aoi[common] - st.doy[common]).mean()}


def hrvpp_check(target, s):
    if hrvpp_end is None or target != "end":
        return {}
    m = pd.concat({"doy": s, "ref": hrvpp_end}, axis=1, join="inner").dropna()
    if len(m) < 4:
        return {"hrvpp_n": len(m)}
    e = m.doy - m.ref
    return {"hrvpp_mae": e.abs().mean(), "hrvpp_bias": e.mean(), "hrvpp_r": m.doy.corr(m.ref), "hrvpp_n": len(m)}


rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # medians / means of candidates without any value
    for (target, method, index, metric), s in candidates.items():
        g = d[(d.method == method) & (d["index"] == index) & (d.metric == metric)]
        gz = g[g.zone != "all"]
        aoi = s.xs("all", level="zone") if "all" in s.index.get_level_values("zone") else pd.Series(dtype=float)
        rows.append({"target": target, "method": method, "index": index, "metric": metric,
                     "coverage": len(gz) / n_zone_years, "observed": (gz.bracket_days <= GAP_DAYS).mean(),
                     "qa_good": gz.qa.isin(["high", "medium"]).mean() if gz.qa.notna().any() else np.nan,
                     "ci_days": (gz.ci_high - gz.ci_low).median() if "ci_high" in gz else np.nan,
                     "order_ok": order_share(target, s.drop(index="all", level="zone", errors="ignore")),
                     "days_per_100m": elevation_slope(s), "aoi_mean_doy": aoi.mean(), "aoi_sd_years": aoi.std(),
                     **meteoswiss_check(target, aoi), **hrvpp_check(target, s)})
comparison = pd.DataFrame(rows).set_index(["target", "method", "index", "metric"]).sort_index()

# ---- short list per date type: mean rank over the criteria a candidate has (rank 1 = best within its date type)
CRITERIA = {"ms_r": False, "hrvpp_mae": True, "observed": False, "order_ok": False, "qa_good": False}   # ascending?
CRITERIA = {c: asc for c, asc in CRITERIA.items() if c in comparison and comparison[c].notna().any()}
ranks = pd.DataFrame({c: comparison.groupby(level="target")[c].rank(ascending=asc) for c, asc in CRITERIA.items()})
comparison["mean_rank"] = ranks.mean(axis=1)
display(comparison.round(2))
print(f"Short list (coverage >= 0.5; mean rank over {', '.join(CRITERIA)}):")
for target in TARGET_METRICS:
    t = comparison.xs(target, level="target")
    top = t[t.coverage >= 0.5].sort_values("mean_rank").head(3)
    for (method, index, metric), r in top.iterrows():
        ms = f", MeteoSwiss r = {r.ms_r:.2f}" if "ms_r" in r and pd.notna(r.ms_r) else ""
        print(f"  {target:5s}  {method} / {index} / {metric}: mean rank {r.mean_rank:.1f}, observed {r.observed:.0%}, "
              f"order {r.order_ok:.0%}{ms}")

# ---- figure: whole-AOI dates per year of every candidate with coverage >= 0.5, with the validation data
fig, axes = plt.subplots(1, len(TARGET_METRICS), figsize=(6 * len(TARGET_METRICS), 5.5), sharex=True)
for ax, target in zip(axes, TARGET_METRICS):
    shown = comparison.xs(target, level="target")
    shown = shown[shown.coverage >= 0.5]
    for k, (method, index, metric) in enumerate(shown.index):
        s = candidates[(target, method, index, metric)]
        if "all" not in s.index.get_level_values("zone"):
            continue
        a = s.xs("all", level="zone")
        ax.plot(a.index + (k - len(shown) / 2) * 0.02, a.values, marker=INDEX_MARKERS.get(index, "o"), ms=5,
                lw=0.7, color=METHOD_COLORS[method], ls="-" if metric != "mid" else ":", alpha=0.8)
    if pheno is not None and target in VALIDATE_WITH:
        st = pheno.xs(VALIDATE_WITH[target], level="param")
        ax.plot(st.index, st.doy, "ks--", lw=2, ms=7,
                label=f"MeteoSwiss {PHENO_PARAMS[VALIDATE_WITH[target]]} (stations, mean {st.height.mean():.0f} m)")
    if hrvpp_end is not None and target == "end" and "all" in hrvpp_end.index.get_level_values("zone"):
        hv = hrvpp_end.xs("all", level="zone")
        ax.plot(hv.index, hv.values, "kD-", lw=2, ms=7, label="HR-VPP end of season")
    ax.set_title(f"{target} (whole AOI)")
    ax.set_xlabel("year")
    ax.grid(alpha=0.3)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=8, loc="upper left")
axes[0].set_ylabel("day of year")
used = comparison[comparison.coverage >= 0.5].reset_index()
handles = ([plt.Line2D([], [], color=METHOD_COLORS[m], lw=2, label=m) for m in dict.fromkeys(used.method)]
           + [plt.Line2D([], [], color="grey", marker=INDEX_MARKERS.get(i, "o"), ls="", label=i)
              for i in dict.fromkeys(used["index"])]
           + [plt.Line2D([], [], color="grey", ls=":", label="mid (50 %) as peak")])
fig.legend(handles=handles, loc="lower center", ncol=8, fontsize=8, bbox_to_anchor=(0.5, -0.06))
plt.suptitle("Season dates of all candidates (coverage >= 0.5) and the validation data")
plt.tight_layout()
plt.show()

## 8. Phenology vs topography (elevation band × aspect)

Only logistic fits with `direction_ok` and at least `MIN_OBS_SIDE` observations on each side of `t0` are used.

- **Plot:** transition DOY against elevation band, one line per aspect (mean ± sd over years).
- **Model:** OLS `doy ~ elev_100m + C(aspect) + C(year)`.
  - The elevation coefficient is in days per 100 m.
  - The aspect coefficient is the north − south difference in days.
  - `C(year)` absorbs year-to-year weather.

**Caveats:**
- Zones share weather and clouds within a year, so the CIs are somewhat optimistic.
- North slopes lose more autumn dates to SCL class 2 (dark / topographic shadow); see `mean_n_obs_*` and `autumn_dark_fraction`.
- PSRI on north slopes in late autumn (low sun) is the most affected.

In [ ]:
autumn = stats_all[stats_all.index.dayofyear >= FIT_WINDOW[0]]
coverage = (zone_table[["elev_band", "aspect", "n_pixels", "elev_mean", "slope_mean", "northness_mean"]]
            .join(logistic_table.groupby(level=["zone", "index"]).n_obs.mean().unstack().add_prefix("mean_n_obs_"))
            .join(autumn.groupby("zone").scl_dark_fraction.mean().rename("autumn_dark_fraction")))
display(coverage.round(2))

zone_attrs = zone_table.drop(index="all")[["elev_band", "aspect", "elev_mean"]]
fits_ok = logistic_table.reset_index().join(zone_attrs, on="zone", how="inner")
fits_ok = fits_ok[fits_ok.direction_ok.eq(True) & (fits_ok.n_before_mid >= MIN_OBS_SIDE)
                  & (fits_ok.n_after_mid >= MIN_OBS_SIDE)]
eos_zones = eos_table.reset_index().join(zone_attrs, on="zone", how="inner")

responses = [(f"{name} logistic mid", fits_ok[fits_ok["index"] == name], "mid_doy") for name in INDICES]
responses.append(("NDVI EOS midpoint", eos_zones, "eos_mid_doy"))

fig, axes = plt.subplots(1, len(responses), figsize=(4 * len(responses), 4), sharex=True)
for ax, (label, data, col) in zip(axes, responses):
    for asp, color in ASPECT_COLORS.items():
        d = data[data.aspect == asp].dropna(subset=[col])
        if d.empty:
            continue
        shift = ELEV_STEP / 2 + {"N": -0.08, "S": 0.08, "flat": 0}[asp] * ELEV_STEP   # band centre, dodged
        ax.scatter(d.elev_band + shift, d[col], s=8, color=color, alpha=0.3)
        g = d.groupby("elev_band")[col].agg(["mean", "std"])
        ax.errorbar(g.index + shift, g["mean"], yerr=g["std"].fillna(0), fmt="o-", color=color, capsize=3,
                    label=asp)
    ax.set_title(label)
    ax.set_xlabel("elevation band centre (m)")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("day of year")
axes[0].legend(title="aspect")
plt.suptitle("Autumn transition vs elevation and aspect (points: zone-years, lines: mean ± sd over years)")
plt.tight_layout()
plt.show()

NS_TERM = "C(aspect, Treatment('S'))[T.N]"


def topo_model(data, col):
    d = data.dropna(subset=[col]).assign(elev_100m=lambda x: x.elev_mean / 100)
    if (not {"N", "S"} <= set(d.aspect) or d.elev_band.nunique() < 2
            or len(d) < d.year.nunique() + d.aspect.nunique() + 3):
        return None
    return smf.ols(f"{col} ~ elev_100m + C(aspect, Treatment('S')) + C(year)", data=d).fit()


rows = []
for label, data, col in responses:
    fit = topo_model(data, col)
    if fit is None:
        print(f"{label}: too few zone-years (needs N and S zones and >= 2 elevation bands), no model")
        continue
    ci = fit.conf_int(ALPHA)
    rows.append({
        "response": label, "n_zone_years": int(fit.nobs), "n_zones": data.zone.nunique(),
        "days_per_100m": fit.params["elev_100m"],
        "elev_ci_low": ci.loc["elev_100m", 0], "elev_ci_high": ci.loc["elev_100m", 1],
        "elev_p": fit.pvalues["elev_100m"],
        "north_minus_south_days": fit.params[NS_TERM],
        "ns_ci_low": ci.loc[NS_TERM, 0], "ns_ci_high": ci.loc[NS_TERM, 1], "ns_p": fit.pvalues[NS_TERM],
        "r2": fit.rsquared,
    })
topo_effects = pd.DataFrame(rows).set_index("response").round(3) if rows else pd.DataFrame()
display(topo_effects)

## 9. Trend / seasonal / remainder decomposition (STL, whole AOI)

Robust STL on a regular 5-day grid splits each index into a long-term trend, a repeating yearly cycle and the remainder. It needs at least two full years.

In [ ]:
STEP_DAYS = 5
PERIOD = int(round(365.25 / STEP_DAYS))   # observations per year on the regular grid


def stl_decompose(series, name):
    s = series.resample(f"{STEP_DAYS}D").mean()
    s = s.loc[s.first_valid_index():s.last_valid_index()].interpolate(method="time")
    if len(s) < 2 * PERIOD:
        print(f"{name}: {len(s) * STEP_DAYS / 365.25:.1f} years of data; STL needs at least 2 full years")
        return None
    res = STL(s, period=PERIOD, robust=True).fit()
    return pd.DataFrame({"observed": s, "trend": res.trend, "seasonal": res.seasonal, "remainder": res.resid})


for name in INDICES:
    dec = stl_decompose(medians[name], name)
    if dec is None:
        continue
    color = INDEX_COLORS[name]
    fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
    axes[0].plot(dec.index, dec.observed, color="grey", lw=1)
    axes[0].set_ylabel(f"{name} observed")
    axes[1].plot(dec.index, dec.trend, color=color, lw=2)
    axes[1].set_ylabel("trend")
    axes[2].plot(dec.index, dec.seasonal, color=color, lw=1.5)
    axes[2].set_ylabel("seasonal")
    axes[3].scatter(dec.index, dec.remainder, s=6, color=color)
    axes[3].set_ylabel("remainder")
    for a in axes[2:]:
        a.axhline(0, color="black", lw=0.5)
    for a in axes:
        a.grid(alpha=0.3)
    axes[0].set_title(f"{name} STL decomposition (period = {PERIOD} x {STEP_DAYS} days)")
    plt.tight_layout()
    plt.show()
    slope = np.polyfit((dec.index - dec.index[0]).days / 365.25, dec.trend.values, 1)[0]
    print(f"{name} trend slope: {slope:+.4f} per year | seasonal amplitude: "
          f"{dec.seasonal.max() - dec.seasonal.min():.3f} | remainder std: {dec.remainder.std():.3f}")

## 10. Export

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_DIR / "ndvi_timeseries_all_dates.csv")
df_clean.to_csv(OUT_DIR / "ndvi_timeseries_clean.csv")
monthly["NDVI"].to_frame("ndvi_monthly_median").to_csv(OUT_DIR / "ndvi_monthly.csv")
season_dates.to_csv(OUT_DIR / "season_dates.csv", index=False)
comparison.to_csv(OUT_DIR / "season_method_comparison.csv")
eos_table.to_csv(OUT_DIR / "eos_by_zone.csv")
logistic_table.to_csv(OUT_DIR / "logistic_by_zone.csv")
zone_table.to_csv(OUT_DIR / "zones.csv")
topo_effects.to_csv(OUT_DIR / "topography_effects.csv")
eos_projection.to_csv(OUT_DIR / "eos_projection.csv")
print(f"Results written to {OUT_DIR}")

## References

- Atzberger, C. & Eilers, P. H. C. (2011). A time series for monitoring vegetation activity and phenology at 10-daily time steps covering large parts of South America. *International Journal of Digital Earth* 4(5), 365–386.
- Beck, P. S. A., Atzberger, C., Høgda, K. A., Johansen, B. & Skidmore, A. K. (2006). Improved monitoring of vegetation dynamics at very high latitudes: a new method using MODIS NDVI. *Remote Sensing of Environment* 100(3), 321–334.
- Busetto, L., Colombo, R., Migliavacca, M., et al. (2010). Remote sensing of larch phenological cycle and analysis of relationships with climate in the Alpine region. *Global Change Biology* 16(9), 2504–2517.
- Copernicus Land Monitoring Service (2021). *HR-VPP Algorithm Theoretical Basis Document: Seasonal Trajectories and VPP parameters*, issue 1.1. https://land.copernicus.eu/en/technical-library
- Eilers, P. H. C. (2003). A perfect smoother. *Analytical Chemistry* 75(14), 3631–3636.
- Gray, J., Sulla-Menashe, D. & Friedl, M. A. (2019). *User Guide to Collection 6 MODIS Land Cover Dynamics (MCD12Q2) Product*. NASA LP DAAC.
- Jönsson, P. & Eklundh, L. (2004). TIMESAT – a program for analyzing time-series of satellite sensor data. *Computers & Geosciences* 30(8), 833–845.
- MeteoSwiss. Phenological observations, open data (`ch.meteoschweiz.ogd-phenology`). https://opendatadocs.meteoswiss.ch/a-data-groundbased/a9-phenological-observations
- Merzlyak, M. N., Gitelson, A. A., Chivkunova, O. B. & Rakitin, V. Y. (1999). Non-destructive optical detection of pigment changes during leaf senescence and fruit ripening. *Physiologia Plantarum* 106(1), 135–141.
- Migliavacca, M., Cremonese, E., Colombo, R., et al. (2008). European larch phenology in the Alps: can we grasp the role of ecological factors by combining field observations and inverse modelling? *International Journal of Biometeorology* 52(7), 587–605.
- Motohka, T., Nasahara, K. N., Oguma, H. & Tsuchida, S. (2010). Applicability of green-red vegetation index for remote sensing of vegetation phenology. *Remote Sensing* 2(10), 2369–2387.
- Richardson, A. D., Hufkens, K., Milliman, T., et al. (2018). Tracking vegetation phenology across diverse North American biomes using PhenoCam imagery. *Scientific Data* 5, 180028.
- Sonnentag, O., Hufkens, K., Teshera-Sterne, C., et al. (2012). Digital repeat photography for phenological research in forest ecosystems. *Agricultural and Forest Meteorology* 152, 159–177.
- Sulik, J. J. & Long, D. S. (2016). Spectral considerations for modeling yield of canola. *Remote Sensing of Environment* 184, 161–174.
- Tian, F., Cai, Z., Jin, H., et al. (2021). Calibrating vegetation phenology from Sentinel-2 using eddy covariance, PhenoCam, and PEP725 networks across Europe. *Remote Sensing of Environment* 260, 112456.
- Zeng, L., Wardlow, B. D., Xiang, D., Hu, S. & Li, D. (2020). A review of vegetation phenological metrics extraction using time-series, multispectral satellite data. *Remote Sensing of Environment* 237, 111511.